# 1. Energy Communities & Consumption Data Preparation

## Overview
This notebook processes and integrates energy consumption data with energy community (ACC/CER) information for the **Área Metropolitana do Porto (AMP)**.

### Inputs
- `input/3-consumos-faturados-por-municipio-ultimos-10-anos.csv` – Monthly consumption data by parish
- `input/comunidades-de-energia.csv` – Energy communities registry

### Outputs
- `dataset_final_parish_amp_FINAL.csv` – Cleaned and merged dataset with features

### Key Features
- **Data Cleaning**: Numeric coercion, Wh→kWh scaling, Portuguese name formatting
- **Aggregation**: Parish-level monthly consumption from voltage-level records
- **Feature Engineering**: Log-transformed energy (`energy_kwh`), temporal features (month, season, quarter), cyclical encoding
- **Energy Community Analysis**: Timeline calculation, control group classification

In [2]:
import pandas as pd
import numpy as np
import os


output_dir = '1_EnergyComunities&Consumption_output'
os.makedirs(output_dir, exist_ok=True)

print("ENERGY COMMUNITIES & CONSUMPTION DATA PREPARATION")

# HELPER FUNCTION: Format Portuguese Names

def format_pt_names(text):
    """Format Portuguese parish names to title case with proper prepositions."""
    # Handle None/NaN (NaN is the only value not equal to itself)
    if text is None or text != text: 
        return text
    
    # Remove quotes and apply title casing
    text = str(text).replace('"', '').title()
    
    # Lowercase Portuguese prepositions
    for prep in [' De ', ' Da ', ' Do ', ' Das ', ' Dos ', ' E ']:
        text = text.replace(prep, prep.lower())
    return text


# STEP 1: Load & Clean Data

print("\n[1/8] Loading data...")
df_cons = pd.read_csv('../input/3-consumos-faturados-por-municipio-ultimos-10-anos.csv', 
                       sep=';', dtype={'DistrictMunicipalityParishCode': str})
df_coms = pd.read_csv('../input/comunidades-de-energia.csv', 
                       sep=';', dtype={'Parish Code': str})

# Strip whitespace from column names
df_coms.columns = df_coms.columns.str.strip()
df_cons.columns = df_cons.columns.str.strip()

# Standardize parish codes (6-digit, zero-padded)
df_cons['parish_code'] = df_cons['DistrictMunicipalityParishCode'].str.zfill(6)
df_coms['parish_code'] = df_coms['Parish Code'].str.zfill(6)

# Convert dates
df_cons['date'] = pd.to_datetime(df_cons['Date'])
df_coms['date'] = pd.to_datetime(df_coms['Date'])

print(f"Consumption records: {len(df_cons):,}")
print(f"Community records: {len(df_coms):,}")


# STEP 1.5: Map & Format Parish Names

print("\n[2/8] Formatting parish names...")
df_coms.rename(columns={'Parish': 'parish_name'}, inplace=True)
df_cons.rename(columns={'parish': 'parish_name_cons'}, inplace=True)

# Apply formatting function
df_coms['parish_name'] = df_coms['parish_name'].apply(format_pt_names)
df_cons['parish_name_cons'] = df_cons['parish_name_cons'].apply(format_pt_names)
df_cons['Municipality'] = df_cons['Municipality'].apply(format_pt_names)

# Use community names as primary source (more reliable)
coms_name_mapping = df_coms.drop_duplicates('parish_code').set_index('parish_code')['parish_name']
df_cons['parish_name'] = df_cons['parish_code'].map(coms_name_mapping).fillna(df_cons['parish_name_cons'])


# STEP 2: Filter for Área Metropolitana do Porto (AMP)

print("\n[3/8] Filtering for Porto Metropolitan Area...")
amp_municipalities = [
    'Arouca', 'Espinho', 'Gondomar', 'Maia', 'Matosinhos', 
    'Oliveira de Azeméis', 'Paredes', 'Porto', 'Póvoa de Varzim', 
    'Santa Maria da Feira', 'Santo Tirso', 'São João da Madeira', 
    'Trofa', 'Vale de Cambra', 'Valongo', 'Vila do Conde', 'Vila Nova de Gaia'
]

df_cons = df_cons[df_cons['Municipality'].isin(amp_municipalities)].copy()
amp_parishes = df_cons['parish_code'].unique()
df_coms = df_coms[df_coms['parish_code'].isin(amp_parishes)].copy()

print(f"AMP municipalities: {len(amp_municipalities)}")
print(f"AMP parishes: {len(amp_parishes)}")


# STEP 3: Process Energy Consumption Data

print("\n[4/8] Processing consumption data...")
# Aggregate by parish, date, and voltage level
df_cons_grouped = df_cons.groupby(['parish_code', 'parish_name', 'Municipality', 'date'])['Energia Ativa'].sum().reset_index()
df_cons_pivot = df_cons.groupby(['parish_code', 'date', 'Voltage level'])['Energia Ativa'].sum().unstack(fill_value=0).reset_index()

# Ensure voltage level columns exist
if 'Baixa Tensão' not in df_cons_pivot.columns: 
    df_cons_pivot['Baixa Tensão'] = 0.0
if 'Muito Alta, Alta e Média Tensões' not in df_cons_pivot.columns: 
    df_cons_pivot['Muito Alta, Alta e Média Tensões'] = 0.0

# Rename to shorter names
df_cons_pivot.rename(columns={
    'Baixa Tensão': 'kwh_BT', 
    'Muito Alta, Alta e Média Tensões': 'kwh_MAT_AT_MT'
}, inplace=True)

# Merge back to grouped data
df_cons_grouped = pd.merge(
    df_cons_grouped[['parish_code', 'parish_name', 'Municipality', 'date']], 
    df_cons_pivot, 
    on=['parish_code', 'date']
)

# Create total energy column and extract temporal features
df_cons_grouped['org_energy'] = df_cons_grouped['kwh_BT'] + df_cons_grouped['kwh_MAT_AT_MT']
df_cons_grouped['energy_kwh'] = np.log1p(df_cons_grouped['org_energy'])  # Log-transformed
df_cons_grouped['year'] = df_cons_grouped['date'].dt.year
df_cons_grouped['month'] = df_cons_grouped['date'].dt.month

print(f"Monthly consumption records: {len(df_cons_grouped):,}")


# STEP 4: Process Energy Communities Data (ACC/CER)

print("\n[5/8] Processing energy community data...")
df_coms_grouped = df_coms.groupby(['parish_code', 'date', 'Type ACC/CER'])['ACC/CER Type Count'].sum().unstack(fill_value=0).reset_index()

# Ensure ACC/CER columns exist
if 'ACC' not in df_coms_grouped.columns: 
    df_coms_grouped['ACC'] = 0.0
if 'CER' not in df_coms_grouped.columns: 
    df_coms_grouped['CER'] = 0.0

df_coms_grouped.rename(columns={'ACC': 'n_acc', 'CER': 'n_cer'}, inplace=True)
df_coms_grouped['n_acc_cer'] = df_coms_grouped['n_acc'] + df_coms_grouped['n_cer']

print(f"Monthly community records: {len(df_coms_grouped):,}")


# STEP 5: Merge Datasets

print("\n[6/8] Merging datasets...")
df_merged = pd.merge(df_cons_grouped, df_coms_grouped, on=['parish_code', 'date'], how='left')

# Fill NaN values in community counts with 0
df_merged['n_acc'] = df_merged['n_acc'].fillna(0).astype(int)
df_merged['n_cer'] = df_merged['n_cer'].fillna(0).astype(int)
df_merged['n_acc_cer'] = df_merged['n_acc_cer'].fillna(0).astype(int)
df_merged = df_merged.sort_values(['parish_code', 'date'])

# Calculate new communities (monthly change)
df_merged['new_acc'] = df_merged.groupby('parish_code')['n_acc'].diff().fillna(df_merged['n_acc']).astype(int).clip(lower=0)
df_merged['new_cer'] = df_merged.groupby('parish_code')['n_cer'].diff().fillna(df_merged['n_cer']).astype(int).clip(lower=0)
df_merged['new_total'] = df_merged['new_acc'] + df_merged['new_cer']

print(f"Merged dataset: {len(df_merged):,} records")


# STEP 6: Calculate Community Timeline

print("\n[7/8] Computing community timeline metrics...")
# Find first date each parish has a community
first_dates = df_coms_grouped[df_coms_grouped['n_acc_cer'] > 0].groupby('parish_code')['date'].min().reset_index()
first_dates.rename(columns={'date': 'acc_cer_start_date'}, inplace=True)

df_merged = pd.merge(df_merged, first_dates, on='parish_code', how='left')

# Calculate months relative to community start
df_merged['months_since_acc_cer'] = (
    (df_merged['date'].dt.year - df_merged['acc_cer_start_date'].dt.year) * 12 +
    (df_merged['date'].dt.month - df_merged['acc_cer_start_date'].dt.month)
)


# STEP 7: Apply Data Quality Rules

print("\n[8/8] Applying data quality filters...")

cols_to_clear = ['acc_cer_start_date', 'new_acc', 'new_cer', 'new_total', 'n_acc', 'n_cer', 'n_acc_cer']
processed_dfs = []

for parish_code, group in df_merged.groupby('parish_code'):
    group = group.sort_values('date').copy()
    group['control'] = False
    needs_recalc = False
    
    if pd.notna(group['acc_cer_start_date'].iloc[0]):
        has_month_zero_or_more = (group['months_since_acc_cer'] >= 0).any()
        
        if not has_month_zero_or_more:
            # RULE 1: Community starts AFTER data period
            needs_recalc = True
            for col in cols_to_clear:
                group[col] = pd.NaT if col == 'acc_cer_start_date' else 0
                
        else:
            # RULE 2: Community has LESS than 12 months of post-creation data
            num_post_months = (group['months_since_acc_cer'] >= 0).sum()
            if num_post_months < 12:
                group = group[group['months_since_acc_cer'] < 0].copy()
                needs_recalc = True
                for col in cols_to_clear:
                    group[col] = pd.NaT if col == 'acc_cer_start_date' else 0
    else:
        # Parish never had a community
        needs_recalc = True
        
    # For invalid communities, treat as control group
    if needs_recalc and not group.empty:
        last_date = group['date'].max()
        group['months_since_acc_cer'] = (
            12 + (group['date'].dt.year - last_date.year) * 12 +
            (group['date'].dt.month - last_date.month)
        ).astype(int)
        group['control'] = True
                    
    processed_dfs.append(group)

df_merged = pd.concat(processed_dfs, ignore_index=True)

# Create binary indicator for valid communities
df_merged['has_acc_cer'] = (df_merged['n_acc_cer'] > 0).astype(int)


# STEP 8: Create Temporal Features

# Cyclical encoding for months (sine/cosine transformation)
df_merged['month_sin'] = np.sin(2 * np.pi * df_merged['month'] / 12)
df_merged['month_cos'] = np.cos(2 * np.pi * df_merged['month'] / 12)
df_merged['quarter'] = df_merged['date'].dt.quarter

def get_season(m):
    """Map month number to season."""
    if m in [12, 1, 2]: return 'Winter'
    elif m in [3, 4, 5]: return 'Spring'
    elif m in [6, 7, 8]: return 'Summer'
    else: return 'Autumn'

df_merged['season'] = df_merged['month'].apply(get_season)


# STEP 9: Prepare Final Output

print("\nPreparing final output...")
final_cols = [
    'parish_code', 'parish_name', 'Municipality', 'date', 'year', 'month',
    'org_energy', 'energy_kwh', 'kwh_BT', 'kwh_MAT_AT_MT', 'acc_cer_start_date', 'has_acc_cer',
    'months_since_acc_cer', 'control', 'new_acc', 'new_cer', 'new_total', 
    'n_acc', 'n_cer', 'n_acc_cer',
    'month_sin', 'month_cos', 'season', 'quarter'
]

df_final = df_merged[final_cols]
df_final = df_final.sort_values(['parish_code', 'date'])


# STEP 10: Save Outputs

print("\nSaving outputs...")
output_csv = '1_EnergyComunities&Consumption_output/dataset_final_parish_amp_FINAL.csv'

df_final.to_csv(output_csv, index=False)

print(f" Saved: {output_csv}")
print(f"\nFinal dataset: {len(df_final):,} records, {df_final['parish_code'].nunique()} parishes")
print("\nDataset preview:")
print(df_final.head())

ENERGY COMMUNITIES & CONSUMPTION DATA PREPARATION

[1/8] Loading data...
Consumption records: 268,520
Community records: 6,636

[2/8] Formatting parish names...

[3/8] Filtering for Porto Metropolitan Area...
AMP municipalities: 17
AMP parishes: 173

[4/8] Processing consumption data...
Monthly consumption records: 10,380

[5/8] Processing energy community data...
Monthly community records: 1,044

[6/8] Merging datasets...
Merged dataset: 10,380 records

[7/8] Computing community timeline metrics...

[8/8] Applying data quality filters...

Preparing final output...

Saving outputs...
 Saved: 1_EnergyComunities&Consumption_output/dataset_final_parish_amp_FINAL.csv

Final dataset: 10,271 records, 173 parishes

Dataset preview:
  parish_code parish_name Municipality       date  year  month  org_energy  \
0      010402   Alvarenga       Arouca 2020-11-01  2020     11  213637.495   
1      010402   Alvarenga       Arouca 2020-12-01  2020     12  233965.863   
2      010402   Alvarenga      